<a target="_blank" href="https://colab.research.google.com/github/agensflow-ai/agensflow-langgraph/blob/main/notebooks/quickstart_adapted.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AgensFlow LangGraph — quickstart (modern-model adapted)

This notebook uses `security_v1_adapted.json` — the paper priors merged with
120 fresh runs on modern models (claude-haiku-4.5, claude-sonnet-5,
thinkingmachines/inkling). Every arm has 25+ real visits; reward means are
calibrated to the specific model tiers listed above.

**Use this notebook if:** you plan to use the same three model tiers the
substrate was adapted on. You get a stronger starting policy (visible in
Section 4 as tighter arm-level differentiation).

For a **model-agnostic** starter with just paper priors, use `quickstart.ipynb`.

**Prerequisites:** `OPENROUTER_API_KEY`. Runtime: ~5-8 min. Cost: ~$1-2.

## 0. Colab setup (auto-skipped if running locally)

One-shot cell: installs the two packages from PyPI, clones the repo to get
the `examples/` directory the graph builder imports from, `cd`s into the
notebooks folder so the relative paths in later cells resolve, and prompts
for `OPENROUTER_API_KEY` if it isn't already in the environment.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '-q',
         'agensflow-mcp', 'agensflow-langgraph',
         'asgi-lifespan', 'langchain-openai', 'python-dotenv'],
        check=True,
    )
    # Clone the repo to get examples/prompts + documents/starter_policies.
    if not os.path.isdir('agensflow-langgraph'):
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/agensflow-ai/agensflow-langgraph.git'],
            check=True,
        )
    os.chdir('agensflow-langgraph/notebooks')
    print(f'  ✓ Colab environment ready — cwd = {os.getcwd()}')

# --- OpenRouter API key setup ---
# Option 1 (preferred): a `.env` file with:
#     OPENROUTER_API_KEY=your_key_here
# Option 2: in-cell magic:
#     %env OPENROUTER_API_KEY=your_key_here
# Option 3: paste when prompted below.

from dotenv import load_dotenv
load_dotenv()
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

if not OPENROUTER_API_KEY:
    print('⚠️  OPENROUTER_API_KEY not found. Set it with %env, a .env file, '
          'or paste it below.')
    OPENROUTER_API_KEY = input('OPENROUTER_API_KEY: ').strip()

os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
print('✅ OPENROUTER_API_KEY loaded')

## 1. Boot the policy server in-process

We use `httpx.ASGITransport` to run the FastAPI app inside this Python
process — same trick our integration tests use. All `/langgraph/*` requests
go through the ASGI transport instead of real HTTP, but everything else
(bandits, storage, tenant isolation) works exactly like a real deployment.

In [ ]:
import os

# Force SQLite-in-memory for this notebook run so we don't need ./data/
os.environ.setdefault('AGF_DATABASE_URL', 'sqlite+aiosqlite:///:memory:')
os.environ.setdefault('AGF_ENV', 'test')
os.environ.setdefault('AGF_JWT_SECRET', 'notebook-not-for-production')

from httpx import ASGITransport, AsyncClient
from asgi_lifespan import LifespanManager

from agensflow_mcp.app import create_app
from agensflow_mcp.db.session import init_db, get_engine
from agensflow_mcp.db.models import Base

app = create_app()
await init_db()
engine = get_engine()
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

lifespan_mgr = LifespanManager(app)
await lifespan_mgr.__aenter__()
server_client = AsyncClient(transport=ASGITransport(app=app), base_url='http://test')
print('  ✓ policy server booted in-process')

## 2. Issue an anonymous API key

The same endpoint a real deployment exposes — `POST /auth/anonymous`.

In [ ]:
resp = await server_client.post('/auth/anonymous')
api_key = resp.json()['api_key']
print(f'  api_key: {api_key[:20]}...')

## 3. Route the adapter's HTTP calls through our in-process server

Normally `agensflow-langgraph`'s client hits a real HTTPS server. Here we
monkey-patch it to use the in-process ASGI transport — one small class
override, then all the decorator's `/langgraph/*` calls flow through the
notebook's own kernel.

In [ ]:
from agensflow_langgraph import client as agf_client

class _NotebookClient(agf_client.AgensFlowClient):
    async def _a_post_model(self, path, payload, model_cls):
        r = await server_client.post(path, json=payload, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

    async def _a_get_model(self, path, model_cls, params=None):
        r = await server_client.get(path, params=params, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

agf_client._CACHE.clear()
agf_client.AgensFlowClient = _NotebookClient
os.environ['AGENSFLOW_SERVER_URL'] = 'http://test'
os.environ['AGENSFLOW_API_KEY'] = api_key
print('  ✓ adapter wired to in-process server')

## 4. Warm-start from adapted policy (`security_v1_adapted.json`)

This starter is the paper priors merged with 120 real runs on modern models —
a stronger starting policy for the SAME model tier bindings this graph uses.
Reward means already reflect what haiku-4.5 / sonnet-5 / inkling produce on
this domain; UCB will spend fewer runs re-exploring before settling.

**Not model-agnostic:** if you swap in different models, the adapted priors
may not match reality. Use `security_v1.json` (via `quickstart.ipynb`) for
the model-agnostic path.

**Isolation guarantee:** `aimport_policy` writes to YOUR tenant scoped by
`api_key` from Section 2. The starter file on disk is read-only.

In [ ]:
from pathlib import Path
from agensflow_langgraph import aimport_policy

# Path resolves whether we're in local notebooks/ or Colab's cloned notebooks/
candidates = [
    Path('..') / 'examples' / 'starter_policies' / 'security_v1_adapted.json',
    Path('/content') / 'agensflow-langgraph' / 'examples' / 'starter_policies' / 'security_v1_adapted.json',
]
policy_path = next((p for p in candidates if p.exists()), None)
if policy_path is None:
    raise FileNotFoundError(
        'security_v1_adapted.json not found. If you\'re a maintainer, generate it via:\n'
        '    python -m examples.security_domain.converge --epochs 6'
    )

result = await aimport_policy(policy_path)
print(f'  ✓ imported {result["signatures_merged"]} signatures / '
      f'{result["actions_merged"]} arms from the paper-trained warm-start')

## 5. Build the security-domain MAS

6-node LangGraph MAS where **every node is decorated with
`@agensflow(pool={...})`**. Each pool declares 2–3 candidate models; the
substrate learns per-node which one wins.

```
  START → planner → memory → solver ─┬─→ critic  ─┐
                                     └─→ verifier ─┴─→ evaluator → END
```

Critic + verifier run in **parallel** — a shape the decorator handles
without special-casing.

### 5a. Import schemas + prompts (from parallel_critic_mas) + security corpus

The schemas and prompts come from `parallel_critic_mas` unchanged — the
security_domain variant only differs in that MEMORY retrieves from a
per-task subset of the security-advisory corpus (12 synthetic CVEs).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from examples.security_domain.prompts import (
    PlannerOutput, MemoryOutput, SolverOutput,
    VerifierOutput, EvaluatorOutput,
    EvidenceItem,
    PLANNER_SYS, MEMORY_SYS, VERIFIER_SYS, EVALUATOR_SYS,
    SOLVER_SYSTEMS,   # dict: {"concise": SYS, "cot": SYS, "evidence": SYS}
    format_planner_input, format_memory_input, format_solver_input,
    format_verifier_input, format_evaluator_input,
)
from examples.security_domain.corpus import CORPUS, get_corpus_subset
from examples.security_domain.tasks import ALL_TASKS
from examples.security_domain.graph import MODEL_BINDING, SKILL_CARDS

def _render_corpus_subset(doc_ids):
    docs = get_corpus_subset(doc_ids) if doc_ids else CORPUS
    return '\n\n'.join(f'[{d.id}]\n{d.text}' for d in docs)

print(f'  ✓ corpus={len(CORPUS)} docs, {len(ALL_TASKS)} paper tasks')
print(f'  ✓ skill cards: {SKILL_CARDS}')
print(f'  ✓ model tier binding: {MODEL_BINDING}')

### 5b. Declare the per-node pools — skill × model factored

This is the paper's action space, ported. **Solver** is a 3×3 matrix of
(skill card × model tier). **Memory** and **verifier** have a `skip` arm the
substrate learned to pick when the stage isn't worth its cost. Every arm
key below has real paper-learned priors from the warm-start in Section 4.

Model tier binding — the arm-key suffix decides the model:
* `haiku` → `anthropic/claude-haiku-4.5`
* `fast`  → `thinkingmachines/inkling`
* `mini`  → `anthropic/claude-sonnet-5`

That leaves the task pool in two families (Anthropic + ThinkingMachines).
The 3-panel judge in Section 8a is xAI + OpenAI + Qwen — fully disjoint.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda

def or_model(model_id, schema=None):
    # default_headers is load-bearing: OpenRouter uses HTTP-Referer + X-Title
    # for app-tier routing. Without them we hit tighter rate limits.
    llm = ChatOpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        model=model_id, temperature=0.0, max_retries=2,
        default_headers={
            'HTTP-Referer': 'https://agensflow.ai',
            'X-Title': 'AgensFlow security_domain (notebook)',
        },
    )
    return llm.with_structured_output(schema, method='function_calling') if schema else llm

def skip_runnable(default):
    """No-op Runnable that returns `default` regardless of input.
    The substrate's 'skip' choice picks this — zero LLM cost."""
    return RunnableLambda(lambda _: default)

pools = {
    'planner':   {'default': or_model(MODEL_BINDING['mini'], PlannerOutput)},
    'memory':    {
        'use':  or_model(MODEL_BINDING['fast'], MemoryOutput),
        'skip': skip_runnable(MemoryOutput(evidence=[])),
    },
    'solver':    {
        f'{skill}-{tier}': or_model(MODEL_BINDING[tier], SolverOutput)
        for skill in SKILL_CARDS for tier in ('haiku', 'fast', 'mini')
    },
    'verifier':  {
        'fast':  or_model(MODEL_BINDING['fast'], VerifierOutput),
        'haiku': or_model(MODEL_BINDING['haiku'], VerifierOutput),
        'skip':  skip_runnable(VerifierOutput(verdict='supported', ungrounded_claims=[])),
    },
    'evaluator': {'default': or_model(MODEL_BINDING['mini'], EvaluatorOutput)},
}
for node, arms in pools.items():
    print(f'  {node:<10} {len(arms):>2} arms: {list(arms)}')

### 5c. Decorate the 6 nodes with `@agensflow`

**This is the whole integration.** Each async node function gets
`@agensflow(pool=pools['name'])` prepended. The decorator:

1. Derives a signature from the node identity + graph context
2. Asks the substrate which arm to use (UCB1 over the pool keys)
3. Injects the chosen `model` into the function body
4. Captures cost + tokens + latency on the returned message
5. POSTs the outcome to `/langgraph/decision/execute` so the bandit updates

Everything after the `@agensflow` line is your ordinary LangGraph node.
The decorator is the only AgensFlow-specific code you write.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from agensflow_langgraph import agensflow
from examples.security_domain.graph import regime_signature, SCENARIO_TO_REGIME

class SecurityMASState(TypedDict, total=False):
    user_task: str; corpus_doc_ids: list[str]
    scenario_class: str; regime: str      # feeds regime_signature
    goal: str; subproblem: str
    evidence: list[dict]
    draft_answer: str; solver_reasoning: str
    verifier_verdict: str; ungrounded_claims: list[str]
    revision_count: int
    final_answer: str; evaluator_reasoning: str
    messages: Annotated[list, add_messages]
    trace: Annotated[list, operator.add]

def _trace(node, model):
    cfg = getattr(model, 'config', None) or {}
    return {'node': node, 'action': (cfg.get('metadata') or {}).get('agensflow_action', '<fell-open>')}

@agensflow(pool=pools['planner'], signature=regime_signature)
async def planner(state, model, config=None):
    r = await model.ainvoke([('system', PLANNER_SYS),
                             ('human', format_planner_input(state['user_task']))])
    return {'goal': r.goal, 'subproblem': r.subproblem, 'trace': [_trace('planner', model)]}

@agensflow(pool=pools['memory'], signature=regime_signature)
async def memory(state, model, config=None):
    r = await model.ainvoke([('system', MEMORY_SYS.format(corpus=_render_corpus_subset(state.get('corpus_doc_ids', [])))),
                             ('human', format_memory_input(state['subproblem']))])
    return {'evidence': [e.model_dump() for e in r.evidence], 'trace': [_trace('memory', model)]}

@agensflow(pool=pools['solver'], signature=regime_signature)
async def solver(state, model, config=None):
    # Solver arm key encodes both skill card and model tier — split it here
    cfg = getattr(model, 'config', None) or {}
    action = (cfg.get('metadata') or {}).get('agensflow_action', 'concise-haiku')
    skill = action.split('-', 1)[0] if '-' in action else 'concise'
    system = SOLVER_SYSTEMS.get(skill, SOLVER_SYSTEMS['concise'])
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', system),
                             ('human', format_solver_input(state['subproblem'], ev))])
    return {'draft_answer': r.draft_answer, 'solver_reasoning': r.reasoning,
            'trace': [_trace('solver', model)]}

@agensflow(pool=pools['verifier'], signature=regime_signature)
async def verifier(state, model, config=None):
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', VERIFIER_SYS),
                             ('human', format_verifier_input(state['subproblem'],
                                                             state['draft_answer'], ev))])
    return {'verifier_verdict': r.verdict, 'ungrounded_claims': list(r.ungrounded_claims),
            'trace': [_trace('verifier', model)]}

def verifier_gate(state):
    if state.get('verifier_verdict') == 'unsupported' and state.get('revision_count', 0) < 2:
        return 'bump_revision'
    return 'evaluator'

async def bump_revision(state):
    return {'revision_count': state.get('revision_count', 0) + 1}

@agensflow(pool=pools['evaluator'], signature=regime_signature)
async def evaluator(state, model, config=None):
    r = await model.ainvoke([('system', EVALUATOR_SYS),
                             ('human', format_evaluator_input(state['goal'], state['draft_answer'],
                                                              state.get('verifier_verdict', 'unknown')))])
    return {'final_answer': r.final_answer, 'evaluator_reasoning': r.merged_reasoning,
            'trace': [_trace('evaluator', model)]}

print('  ✓ 5 decorated nodes defined + verifier-gate + bump_revision')

### 5d. Wire the StateGraph with the verifier gate

Linear topology + a conditional edge from `verifier`. If the verifier says
`unsupported` and we've used less than 2 revision budgets, we send state
back through `bump_revision → solver`. Otherwise we proceed to `evaluator`.

In [ ]:
graph = StateGraph(SecurityMASState)
graph.add_node('planner',       planner)
graph.add_node('memory',        memory)
graph.add_node('solver',        solver)
graph.add_node('verifier',      verifier)
graph.add_node('bump_revision', bump_revision)
graph.add_node('evaluator',     evaluator)

graph.add_edge(START,           'planner')
graph.add_edge('planner',       'memory')
graph.add_edge('memory',        'solver')
graph.add_edge('solver',        'verifier')
graph.add_conditional_edges(
    'verifier', verifier_gate,
    {'bump_revision': 'bump_revision', 'evaluator': 'evaluator'},
)
graph.add_edge('bump_revision', 'solver')
graph.add_edge('evaluator',     END)

compiled = graph.compile(checkpointer=InMemorySaver())
print('  ✓ graph compiled')
print(f'  nodes: {list(compiled.get_graph().nodes)}')

## 6. What did the substrate actually learn?

The imported policy carries reward-mean rankings per (node, arm). Reading them
gives you the substrate's current preference *before* running any task. If the
paper-adapted starter is honest about modern models, the top-ranked arms
should look sensible.

In [ ]:
from collections import defaultdict
from agensflow_langgraph.client import get_client

policy = (await get_client().a_export_policy()).policy

# Signatures are `f"{node}:{regime}"` — group by node prefix
by_node = defaultdict(dict)
for sig, arms in policy.items():
    if ':' in sig:
        node, regime = sig.split(':', 1)
    else:
        node, regime = sig, 'default'
    by_node[node][regime] = arms

print('  === Current per-(node, regime) arm rankings ===\n')
for node in ('planner', 'memory', 'solver', 'verifier', 'evaluator'):
    regimes = by_node.get(node, {})
    if not regimes:
        continue
    print(f'  {node}:')
    for regime in sorted(regimes):
        arms = regimes[regime]
        ranked = sorted(arms.items(), key=lambda kv: -kv[1].get('reward_mean', 0))
        print(f'    [{regime}]')
        for arm, stats in ranked:
            v = int(stats.get('visits', 0))
            m = stats.get('reward_mean', 0)
            marker = ' ← preferred' if arm == ranked[0][0] else ''
            print(f'      {arm:<18} v={v:>3}  μ={m:.3f}{marker}')
    print()

## 7. Per-task-class routing on real tasks

Run three tasks from *different* scenario classes. If the substrate learned
per-task routing (not just "one best arm"), you'll see it pick different
arms for different task shapes:

* **C1.1** — single-doc factual lookup (should route to a cheap/fast arm)
* **C7.1** — multi-step reasoning (should route to a `cot-*` arm)
* **C3.1** — cross-doc consistency check (should route to an `evidence-*` arm)

Same substrate, three tasks, no re-training.

In [ ]:
import time

demo_task_ids = ['C1.1', 'C7.1', 'C3.1']
demo_tasks = [next(t for t in ALL_TASKS if t.id == tid) for tid in demo_task_ids]
demo_records = []

for t in demo_tasks:
    print(f'  ── Task {t.id} (class {t.scenario_class}) ──')
    print(f'  Q: {t.user_task[:110]}{"..." if len(t.user_task) > 110 else ""}\n')
    t0 = time.monotonic()
    r = await compiled.ainvoke(
        {'user_task': t.user_task, 'corpus_doc_ids': t.corpus_doc_ids, 'scenario_class': t.scenario_class, 'trace': []},
        config={'configurable': {'thread_id': f'section7_{t.id}'}},
    )
    elapsed = time.monotonic() - t0
    routing = {step['node']: step['action'] for step in r['trace']}
    demo_records.append({'task': t, 'routing': routing, 'answer': r['final_answer'], 'elapsed_s': elapsed})
    print(f'  A: {r["final_answer"][:180]}...\n')
    print(f'  Routing: ' + ' → '.join(f'{n}={routing.get(n, "?")}' for n in ('planner', 'memory', 'solver', 'verifier', 'evaluator')))
    print(f'  Wall clock: {elapsed:.1f}s\n')

print('  === Per-task routing summary ===')
print(f'  {"task":<8} {"class":<7} ' + ' '.join(f'{n:<12}' for n in ('planner', 'memory', 'solver', 'verifier', 'evaluator')))
for rec in demo_records:
    routing = rec['routing']
    print(f'  {rec["task"].id:<8} {rec["task"].scenario_class:<7} ' +
          ' '.join(f'{routing.get(n, "?"):<12}' for n in ('planner', 'memory', 'solver', 'verifier', 'evaluator')))

## 8. Cost vs quality — substrate vs fixed baselines

Take **one representative task** (C7.1, multi-step reasoning) and run it
through three strategies:

1. **Substrate-routed** — what the notebook just did (per-node arm selection from the imported policy)
2. **All-cheapest** — every node pinned to `thinkingmachines/inkling`, no substrate
3. **All-most-capable** — every node pinned to `anthropic/claude-sonnet-5`, no substrate

For each: total tokens, wall clock, and the 3-panel judge quality. This is the
Pareto question — does the substrate land near "all-most-capable" quality at
a fraction of the cost?

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from agensflow_langgraph.judge_panel import relative_quality
from agensflow_langgraph.callbacks import CostCapture
import json as _json, re as _re, time as _time

# --- Approx OpenRouter pricing (per 1M tokens) --------------------------- #
PRICING = {
    'anthropic/claude-haiku-4.5':  {'in': 0.80, 'out': 4.00},
    'anthropic/claude-sonnet-5':   {'in': 3.00, 'out': 15.00},
    'thinkingmachines/inkling':    {'in': 0.15, 'out': 0.60},
    'openai/gpt-5.4-nano':         {'in': 0.10, 'out': 0.40},
    'openai/gpt-5.4-mini':         {'in': 0.30, 'out': 1.20},
}
def _approx_cost(model_id, tokens_in, tokens_out):
    p = PRICING.get(model_id)
    if not p: return 0.0
    return (tokens_in * p['in'] + tokens_out * p['out']) / 1_000_000

def _arm_to_model(signature, arm):
    node = signature.split(':', 1)[0] if signature else signature
    if arm == 'skip': return None
    if node in ('planner', 'evaluator'): return 'anthropic/claude-sonnet-5'
    if node == 'memory': return 'thinkingmachines/inkling'
    if node == 'verifier':
        return {'fast': 'thinkingmachines/inkling',
                'haiku': 'anthropic/claude-haiku-4.5'}.get(arm)
    if node == 'solver':
        tier = arm.split('-', 1)[-1] if '-' in arm else arm
        return {'haiku': 'anthropic/claude-haiku-4.5',
                'fast':  'thinkingmachines/inkling',
                'mini':  'anthropic/claude-sonnet-5'}.get(tier)
    return None

# --- Pinned baseline with 3-way structured-output fallback --------------- #
def _pinned(model_id, schema, *, solver_skill='concise', memory_skip=False, verifier_skip=False):
    llm = ChatOpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        model=model_id, temperature=0.0, max_retries=2,
        default_headers={'HTTP-Referer': 'https://agensflow.ai',
                         'X-Title': 'AgensFlow demo baselines'},
    )
    fc = llm.with_structured_output(schema, method='function_calling')
    js = llm.with_structured_output(schema, method='json_schema')
    async def _raw_parse(msgs, config=None):
        resp = await llm.ainvoke(msgs, config=config)
        text = getattr(resp, 'content', None) or str(resp)
        m = _re.search(r'\{.*\}', text, _re.DOTALL)
        if not m: raise ValueError(f'no JSON in raw response: {text[:200]}')
        return schema.model_validate(_json.loads(m.group(0)))
    class _Shim:
        async def ainvoke(self, msgs, config=None):
            try: return await fc.ainvoke(msgs, config=config)
            except Exception as e_fc:
                try: return await js.ainvoke(msgs, config=config)
                except Exception:
                    try: return await _raw_parse(msgs, config=config)
                    except Exception: raise e_fc
    return _Shim()

def _build_pinned(model_id, *, solver_skill='concise', memory_skip=False, verifier_skip=False):
    p, m, s, v, e = (_pinned(model_id, sch, solver_skill=solver_skill,
                             memory_skip=memory_skip, verifier_skip=verifier_skip)
                     for sch in (PlannerOutput, MemoryOutput, SolverOutput, VerifierOutput, EvaluatorOutput))
    async def _planner(state, config=None):
        r = await p.ainvoke([('system', PLANNER_SYS), ('human', format_planner_input(state['user_task']))], config=config)
        return {'goal': r.goal, 'subproblem': r.subproblem}
    async def _memory(state, config=None):
        if memory_skip: return {'evidence': []}
        r = await m.ainvoke([('system', MEMORY_SYS.format(corpus=_render_corpus_subset(state.get('corpus_doc_ids', [])))),
                             ('human', format_memory_input(state['subproblem']))], config=config)
        return {'evidence': [ev.model_dump() for ev in r.evidence]}
    async def _solver(state, config=None):
        ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
        system = SOLVER_SYSTEMS.get(solver_skill, SOLVER_SYSTEMS['concise'])
        r = await s.ainvoke([('system', system),
                             ('human', format_solver_input(state['subproblem'], ev))], config=config)
        return {'draft_answer': r.draft_answer, 'solver_reasoning': r.reasoning}
    async def _verifier(state, config=None):
        if verifier_skip: return {'verifier_verdict': 'supported', 'ungrounded_claims': []}
        ev = [EvidenceItem(**el) for el in state.get('evidence', [])]
        r = await v.ainvoke([('system', VERIFIER_SYS),
                             ('human', format_verifier_input(state['subproblem'], state['draft_answer'], ev))], config=config)
        return {'verifier_verdict': r.verdict, 'ungrounded_claims': list(r.ungrounded_claims)}
    async def _evaluator(state, config=None):
        r = await e.ainvoke([('system', EVALUATOR_SYS),
                             ('human', format_evaluator_input(state['goal'], state['draft_answer'], state.get('verifier_verdict', 'unknown')))], config=config)
        return {'final_answer': r.final_answer, 'evaluator_reasoning': r.merged_reasoning}
    g = StateGraph(SecurityMASState)
    for n, fn in [('planner', _planner), ('memory', _memory), ('solver', _solver),
                  ('verifier', _verifier), ('evaluator', _evaluator)]:
        g.add_node(n, fn)
    for a, b in [(START, 'planner'), ('planner', 'memory'), ('memory', 'solver'),
                 ('solver', 'verifier'), ('verifier', 'evaluator'), ('evaluator', END)]:
        g.add_edge(a, b)
    return g.compile(checkpointer=InMemorySaver())

# --- Setup ---------------------------------------------------------------- #
representative = next(t for t in ALL_TASKS if t.id == 'C7.1')
print(f'  Task: {representative.id} — {representative.user_task[:120]}...\n')
pinned_cheap = _build_pinned('thinkingmachines/inkling')
pinned_best  = _build_pinned('anthropic/claude-sonnet-5')

baseline_llm = ChatOpenAI(
    base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'],
    model='thinkingmachines/inkling', temperature=0.0,
    default_headers={'HTTP-Referer': 'https://agensflow.ai', 'X-Title': 'AgensFlow demo'},
)
baseline_answer = (await baseline_llm.ainvoke([('system', 'Answer concisely.'),
                                               ('human', representative.user_task)])).content
PANEL_MODELS = ('x-ai/grok-4.3', 'openai/gpt-5.4-mini', 'qwen/qwen3.6-flash')

async def _substrate_cost_from_server(thread_id):
    resp = await server_client.get('/langgraph/decisions?limit=50',
                                   headers={'Authorization': f'Bearer {api_key}'})
    recent = resp.json().get('decisions', [])[:5]
    cost = 0.0; t_in = 0; t_out = 0
    for d in recent:
        ti = int(d.get('tokens_input') or 0); to = int(d.get('tokens_output') or 0)
        model = _arm_to_model(d.get('signature', ''), d.get('action', ''))
        cost += _approx_cost(model, ti, to) if model else 0.0
        t_in += ti; t_out += to
    return cost, t_in, t_out

strategies = [
    ('substrate',    compiled,      None),
    ('all-cheapest', pinned_cheap,  'thinkingmachines/inkling'),
    ('all-best',     pinned_best,   'anthropic/claude-sonnet-5'),
]

rows = []
substrate_routing = {}
for name, graph, model_id in strategies:
    cap = CostCapture()
    tid = f'section8_{name}'
    t0 = _time.monotonic()
    try:
        r = await graph.ainvoke(
            {'user_task': representative.user_task,
             'corpus_doc_ids': representative.corpus_doc_ids,
             'scenario_class': representative.scenario_class,
             'trace': [], 'revision_count': 0},
            config={'configurable': {'thread_id': tid}, 'callbacks': [cap]},
        )
    except Exception as ex:
        print(f'  {name:<14} ERROR: {type(ex).__name__}: {str(ex)[:120]}')
        rows.append({'strategy': name, 'quality': None, 'cost_usd': 0.0,
                     'tokens_in': 0, 'tokens_out': 0, 'latency_s': 0.0, 'error': str(ex)[:200]})
        continue
    elapsed = _time.monotonic() - t0
    if name == 'substrate':
        substrate_routing = {step['node']: step['action'] for step in r.get('trace', [])}
        cost, t_in, t_out = await _substrate_cost_from_server(tid)
    else:
        t_in, t_out = cap.input_tokens, cap.output_tokens
        cost = _approx_cost(model_id, t_in, t_out) if model_id else cap.cost_usd
    q, _axes = await relative_quality(
        task=representative.user_task, candidate=r['final_answer'], baseline=baseline_answer,
        openrouter_key=os.environ['OPENROUTER_API_KEY'], models=PANEL_MODELS,
    )
    rows.append({'strategy': name, 'quality': q, 'tokens_in': t_in, 'tokens_out': t_out,
                 'cost_usd': cost, 'latency_s': elapsed, 'answer_len': len(r['final_answer'])})
    print(f'  {name:<14} q={q:.3f}  tokens=in{t_in}/out{t_out}  cost=${cost:.4f}  lat={elapsed:.1f}s')

# --- best-fair: sonnet-5 with substrate's skip+skill picks --------------- #
if substrate_routing:
    memory_skip = substrate_routing.get('memory') == 'skip'
    verifier_skip = substrate_routing.get('verifier') == 'skip'
    solver_arm = substrate_routing.get('solver', 'concise-mini')
    solver_skill = solver_arm.split('-', 1)[0] if '-' in solver_arm else 'concise'
    fair = _build_pinned('anthropic/claude-sonnet-5',
                         solver_skill=solver_skill, memory_skip=memory_skip, verifier_skip=verifier_skip)
    print(f'\n  [fair-attribution: sonnet-5 everywhere, substrate\'s skill+skip picks: '
          f'solver_skill={solver_skill}, memory_skip={memory_skip}, verifier_skip={verifier_skip}]')
    cap = CostCapture(); t0 = _time.monotonic()
    try:
        r = await fair.ainvoke(
            {'user_task': representative.user_task,
             'corpus_doc_ids': representative.corpus_doc_ids,
             'scenario_class': representative.scenario_class,
             'trace': [], 'revision_count': 0},
            config={'configurable': {'thread_id': 'section8_best-fair'}, 'callbacks': [cap]},
        )
        elapsed = _time.monotonic() - t0
        cost = _approx_cost('anthropic/claude-sonnet-5', cap.input_tokens, cap.output_tokens)
        q, _ = await relative_quality(
            task=representative.user_task, candidate=r['final_answer'], baseline=baseline_answer,
            openrouter_key=os.environ['OPENROUTER_API_KEY'], models=PANEL_MODELS,
        )
        rows.append({'strategy': 'best-fair', 'quality': q, 'tokens_in': cap.input_tokens,
                     'tokens_out': cap.output_tokens, 'cost_usd': cost, 'latency_s': elapsed,
                     'answer_len': len(r['final_answer'])})
        print(f'  {"best-fair":<14} q={q:.3f}  tokens=in{cap.input_tokens}/out{cap.output_tokens}  '
              f'cost=${cost:.4f}  lat={elapsed:.1f}s')
    except Exception as ex:
        print(f'  {"best-fair":<14} ERROR: {type(ex).__name__}: {str(ex)[:120]}')
        rows.append({'strategy': 'best-fair', 'quality': None, 'cost_usd': 0.0,
                     'tokens_in': 0, 'tokens_out': 0, 'latency_s': 0.0, 'error': str(ex)[:200]})

# --- Pareto table + attribution ------------------------------------------ #
best_row = next((r for r in rows if r['strategy'] == 'all-best' and r.get('quality') is not None), None)
print('\n  === Pareto: substrate vs baselines ===')
print(f'  {"strategy":<14} {"quality":>8} {"cost":>10} {"latency":>9}' + (f' {"q%":>6} {"$%":>6}' if best_row else ''))
for r in rows:
    if r.get('quality') is None:
        print(f'  {r["strategy"]:<14} (failed: {r.get("error", "unknown")[:80]})')
        continue
    line = f'  {r["strategy"]:<14} {r["quality"]:>8.3f} ${r["cost_usd"]:>9.4f} {r["latency_s"]:>8.1f}s'
    if best_row:
        q_pct = 100 * r['quality'] / max(best_row['quality'], 1e-6)
        c_pct = 100 * r['cost_usd'] / max(best_row['cost_usd'], 1e-9)
        line += f' {q_pct:>5.1f}% {c_pct:>5.1f}%'
    print(line)

print('\n  ── What the columns mean ──')
print('    substrate    : full learned routing (model tier + skill card + skip)')
print('    all-cheapest : inkling everywhere, always-use, concise skill')
print('    all-best     : sonnet-5 everywhere, always-use, concise skill')
print('    best-fair    : sonnet-5 everywhere, but substrate\'s skip + skill choices')
print('                   (isolates model-tier savings from skip+skill savings)')

sub = next((r for r in rows if r['strategy'] == 'substrate' and r.get('quality') is not None), None)
fair_row = next((r for r in rows if r['strategy'] == 'best-fair' and r.get('quality') is not None), None)
cheap = next((r for r in rows if r['strategy'] == 'all-cheapest' and r.get('quality') is not None), None)
print('\n  ── Attribution ──')
if sub and best_row:
    saved_c = 1 - sub['cost_usd'] / max(best_row['cost_usd'], 1e-9)
    q_delta = (sub['quality'] - best_row['quality']) / max(best_row['quality'], 1e-6)
    q_verb = 'gains' if q_delta >= 0 else 'trades off'
    print(f'    substrate vs all-best (naive): saves {saved_c*100:.0f}% cost, {q_verb} {abs(q_delta)*100:.0f}% quality')
if best_row and fair_row:
    skip_saved = 1 - fair_row['cost_usd'] / max(best_row['cost_usd'], 1e-9)
    print(f'    ├─ from skip+skill picks alone: saves {skip_saved*100:.0f}% cost (best-fair vs all-best)')
if sub and fair_row:
    tier_saved = 1 - sub['cost_usd'] / max(fair_row['cost_usd'], 1e-9)
    print(f'    └─ from model-tier picks alone: saves {tier_saved*100:.0f}% cost (substrate vs best-fair)')
if sub and cheap:
    q_gain = (sub['quality'] - cheap['quality']) / max(cheap['quality'], 1e-6)
    print(f'    substrate vs all-cheapest: {q_gain*100:+.0f}% quality (spends ${sub["cost_usd"] - cheap["cost_usd"]:.4f} more)')
print('\n  ── Caveat ──')
print('    Single task, single trial. Substrate performance vs baselines depends')
print('    heavily on task shape + judge stability. This demo is an existence proof')
print('    of the routing story, not a benchmark result.')

## 9. Cleanup

Close the in-process server.

In [ ]:
await server_client.aclose()
await lifespan_mgr.__aexit__(None, None, None)
print('  ✓ done')

## What you just saw

1. **Section 4** — imported a bandit policy carrying rank-order priors from
   the paper's security-domain training work.
2. **Sections 5** — defined a 6-node LangGraph MAS with `@agensflow` on every
   node. The decorator is the entire integration surface.
3. **Section 6** — inspected the substrate's current arm rankings without
   running any task. Real prior knowledge, visible.
4. **Section 7** — ran 3 tasks from different scenario classes; the substrate
   picked different arms per class based on the learned priors.
5. **Section 8** — Pareto comparison: substrate-routed vs all-cheapest vs
   all-most-capable on the same task. The substrate's value is in the
   quality-per-dollar column.